# VIII — Generación de Datos Sintéticos: Withdrawn & Fail

**Objetivo**: Generar casos sintéticos de estudiantes con resultado `Withdrawn` y `Fail` para aumentar las clases minoritarias en el dataset de entrenamiento.

**Estrategia dual**:
- **Método 1 — GMM** (Gaussian Mixture Model): rápido, interpretable, buen modelado de distribuciones multimodales
- **Método 2 — VAE** (Variational Autoencoder): captura relaciones no lineales complejas en el espacio latente

**Output**: `data/2.1_sintetic_data/`
- `synthetic_gmm_withdrawn.csv`
- `synthetic_gmm_fail.csv`
- `synthetic_vae_withdrawn.csv`
- `synthetic_vae_fail.csv`
- `synthetic_combined.csv` — fusión de ambos métodos

## 0. Imports & Configuración

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Sklearn
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

# TensorFlow/Keras para VAE
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Configuración visual
plt.style.use('dark_background')
PALETTE = {'Withdrawn': '#FF6B6B', 'Fail': '#FFA500', 'Real': '#4ECDC4', 'Synthetic': '#FFE66D'}
np.random.seed(42)
tf.random.set_seed(42)

# Rutas
BASE_DIR = Path('/workspace/TFM_education_ai_analytics')
DATA_TRAIN = BASE_DIR / 'data' / '2_processed' / 'training'
OUTPUT_DIR = BASE_DIR / 'data' / '2.1_sintetic_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ Output dir: {OUTPUT_DIR}')
print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

2026-03-03 11:32:18.171577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Output dir: /workspace/TFM_education_ai_analytics/data/2.1_sintetic_data
✅ TensorFlow version: 2.20.0
✅ GPU disponible: True


## 1. Carga de Datos

In [2]:
# Cargar los 3 ficheros
students_df     = pd.read_csv(DATA_TRAIN / 'students.csv')
assessments_df  = pd.read_csv(DATA_TRAIN / 'assessments.csv')
interactions_df = pd.read_csv(DATA_TRAIN / 'interactions.csv')

print(f'Students:     {students_df.shape}')
print(f'Assessments:  {assessments_df.shape}')
print(f'Interactions: {interactions_df.shape}')
print()
print('Distribución final_result:')
print(students_df['final_result'].value_counts())
print()
print('Students columns:', students_df.columns.tolist())

Students:     (22785, 15)
Assessments:  (121261, 10)
Interactions: (7474712, 9)

Distribución final_result:
final_result
Pass           8620
Withdrawn      7105
Fail           4948
Distinction    2112
Name: count, dtype: int64

Students columns: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'date_unregistration', 'module_presentation_length']


## 2. Feature Engineering — Agregar Features por Estudiante

In [3]:
# ── 2.1 Features de Assessments ────────────────────────────────────────────
# Agrupar por estudiante: nota media, min, max, std, n_assessments, n_banked
agg_assess = assessments_df.groupby('id_student').agg(
    score_mean   = ('score', 'mean'),
    score_min    = ('score', 'min'),
    score_max    = ('score', 'max'),
    score_std    = ('score', 'std'),
    n_assess     = ('id_assessment', 'count'),
    n_banked     = ('is_banked', 'sum'),
    days_late_mean = ('date_submitted', 'mean'),  # proxy de puntualidad
).reset_index()

agg_assess['score_std'] = agg_assess['score_std'].fillna(0)
print('Assessment features shape:', agg_assess.shape)
agg_assess.head(3)

Assessment features shape: (16347, 8)


,id_student,score_mean,score_min,score_max,score_std,n_assess,n_banked,days_late_mean
0,6516,61.8,48.0,77.0,10.329569,5,0,111.6
1,8462,87.0,83.0,93.0,4.472136,7,4,23.0
2,11391,82.0,78.0,85.0,3.082207,5,0,112.4


In [4]:
# ── 2.2 Features de Interactions ───────────────────────────────────────────
# Agrupar por estudiante: clics totales, días activos, tipos únicos de actividad
agg_inter = interactions_df.groupby('id_student').agg(
    total_clicks       = ('sum_click', 'sum'),
    mean_clicks_per_day= ('sum_click', 'mean'),
    max_clicks_day     = ('sum_click', 'max'),
    active_days        = ('date', pd.Series.nunique),
    n_activity_types   = ('activity_type', pd.Series.nunique),
    n_interactions     = ('sum_click', 'count'),
).reset_index()

# Clicks por tipo de actividad (top 5 más frecuentes)
top_activities = interactions_df['activity_type'].value_counts().head(5).index.tolist()
for act in top_activities:
    mask = interactions_df['activity_type'] == act
    act_clicks = interactions_df[mask].groupby('id_student')['sum_click'].sum().rename(f'clicks_{act}')
    agg_inter = agg_inter.merge(act_clicks, on='id_student', how='left')
    agg_inter[f'clicks_{act}'] = agg_inter[f'clicks_{act}'].fillna(0)

print('Interaction features shape:', agg_inter.shape)
print('Top activities:', top_activities)
agg_inter.head(3)

Interaction features shape: (18233, 12)
Top activities: ['forumng', 'oucontent', 'subpage', 'homepage', 'quiz']


,id_student,total_clicks,mean_clicks_per_day,max_clicks_day,active_days,n_activity_types,n_interactions,clicks_forumng,clicks_oucontent,clicks_subpage,clicks_homepage,clicks_quiz
0,6516,2791,4.216012,49,159,7,662,451.0,1505.0,143.0,497.0,0.0
1,8462,656,2.157895,16,56,9,304,38.0,64.0,227.0,191.0,0.0
2,11391,934,4.765306,76,40,6,196,193.0,553.0,32.0,138.0,0.0


In [5]:
# ── 2.3 Merge Feature Dataset ───────────────────────────────────────────────
# Codificar variables categóricas de students
cat_cols = ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']
students_enc = students_df.copy()

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    students_enc[col] = le.fit_transform(students_enc[col].astype(str))
    le_dict[col] = le

# Merge todo
full_df = students_enc.merge(agg_assess, on='id_student', how='left')
full_df = full_df.merge(agg_inter, on='id_student', how='left')

# Fill NAs: estudiantes sin assessments o sin interacciones
numeric_cols = full_df.select_dtypes(include=[np.number]).columns
full_df[numeric_cols] = full_df[numeric_cols].fillna(0)

print('Dataset completo shape:', full_df.shape)
print('Distribución:')
print(full_df['final_result'].value_counts())
full_df.head(3)

Dataset completo shape: (22785, 33)
Distribución:
final_result
Pass           8620
Withdrawn      7105
Fail           4948
Distinction    2112
Name: count, dtype: int64


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,mean_clicks_per_day,max_clicks_day,active_days,n_activity_types,n_interactions,clicks_forumng,clicks_oucontent,clicks_subpage,clicks_homepage,clicks_quiz
0,AAA,2013J,11391,1,0,1,9,2,0,240,...,4.765306,76.0,40.0,6.0,196.0,193.0,553.0,32.0,138.0,0.0
1,AAA,2013J,28400,0,6,1,2,1,0,60,...,3.337209,23.0,80.0,7.0,430.0,417.0,537.0,87.0,324.0,0.0
2,AAA,2013J,32885,0,11,2,5,0,0,60,...,2.937500,22.0,70.0,7.0,352.0,194.0,494.0,79.0,204.0,0.0


## 3. Filtrar Withdrawn y Fail — Preparar para Generación

In [6]:
# Features numéricas para modelado (excluir IDs y target)
EXCLUDE_COLS = ['id_student', 'code_module', 'code_presentation', 'final_result',
                'date_registration', 'date_unregistration']
FEATURE_COLS = [c for c in full_df.columns if c not in EXCLUDE_COLS]

print(f'Features utilizadas ({len(FEATURE_COLS)}):')
print(FEATURE_COLS)

# Separar por clase
df_withdrawn = full_df[full_df['final_result'] == 'Withdrawn'][FEATURE_COLS].copy().reset_index(drop=True)
df_fail      = full_df[full_df['final_result'] == 'Fail'][FEATURE_COLS].copy().reset_index(drop=True)

print(f'\nWithdrawn reales: {len(df_withdrawn)}')
print(f'Fail reales:      {len(df_fail)}')

# Cuántos sintéticos generar (equiparar con Pass)
n_pass = (full_df['final_result'] == 'Pass').sum()
n_synth_withdrawn = max(0, n_pass - len(df_withdrawn))
n_synth_fail      = max(0, n_pass - len(df_fail))
print(f'\nPass reales:           {n_pass}')
print(f'Sintéticos Withdrawn:  {n_synth_withdrawn}')
print(f'Sintéticos Fail:       {n_synth_fail}')

Features utilizadas (27):
['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'module_presentation_length', 'score_mean', 'score_min', 'score_max', 'score_std', 'n_assess', 'n_banked', 'days_late_mean', 'total_clicks', 'mean_clicks_per_day', 'max_clicks_day', 'active_days', 'n_activity_types', 'n_interactions', 'clicks_forumng', 'clicks_oucontent', 'clicks_subpage', 'clicks_homepage', 'clicks_quiz']

Withdrawn reales: 7105
Fail reales:      4948

Pass reales:           8620
Sintéticos Withdrawn:  1515
Sintéticos Fail:       3672


In [7]:
# ── Escalar features ────────────────────────────────────────────────────────
scaler_w = StandardScaler()
scaler_f = StandardScaler()

X_withdrawn_scaled = scaler_w.fit_transform(df_withdrawn.values.astype(np.float32))
X_fail_scaled      = scaler_f.fit_transform(df_fail.values.astype(np.float32))

print('X_withdrawn_scaled:', X_withdrawn_scaled.shape)
print('X_fail_scaled:     ', X_fail_scaled.shape)

X_withdrawn_scaled: (7105, 27)
X_fail_scaled:      (4948, 27)


## 4. Método 1 — Gaussian Mixture Model (GMM)

Ventajas:
- Modela distribuciones multimodales (distintos perfiles de estudiante que abandona)
- Genera muestras directamente del espacio de features
- Rápido y reproducible

In [8]:
def find_best_gmm(X, max_components=10, covariance_type='full'):
    """Seleccionar nº de componentes GMM por BIC."""
    bic_scores = []
    n_range = range(2, min(max_components + 1, len(X) // 10))
    for n in n_range:
        gmm = GaussianMixture(n_components=n, covariance_type=covariance_type,
                               random_state=42, max_iter=200)
        gmm.fit(X)
        bic_scores.append((n, gmm.bic(X), gmm))
    best = min(bic_scores, key=lambda x: x[1])
    print(f'  Mejor n_components={best[0]} | BIC={best[1]:.1f}')
    return best[2], bic_scores

print('=== GMM para Withdrawn ===')
gmm_withdrawn, bic_w = find_best_gmm(X_withdrawn_scaled, max_components=8)

print('\n=== GMM para Fail ===')
gmm_fail, bic_f = find_best_gmm(X_fail_scaled, max_components=8)

=== GMM para Withdrawn ===


ValueError: Fitting the mixture model failed because some components have ill-defined empirical covariance (for instance caused by singleton or collapsed samples). Try to decrease the number of components, increase reg_covar, or scale the input data. The numerical accuracy can also be improved by passing float64 data instead of float32.

In [ ]:
# Plot BIC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, bic_scores, title in zip(axes, [bic_w, bic_f], ['Withdrawn', 'Fail']):
    ns   = [b[0] for b in bic_scores]
    bics = [b[1] for b in bic_scores]
    ax.plot(ns, bics, 'o-', color=PALETTE[title], linewidth=2)
    best_n = ns[np.argmin(bics)]
    ax.axvline(best_n, color='white', linestyle='--', alpha=0.6, label=f'Óptimo: {best_n}')
    ax.set_title(f'GMM BIC — {title}', fontsize=13, pad=10)
    ax.set_xlabel('Nº Componentes'); ax.set_ylabel('BIC')
    ax.legend()
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gmm_bic_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: gmm_bic_selection.png')

In [ ]:
# ── Generar muestras sintéticas con GMM ────────────────────────────────────
def generate_gmm_samples(gmm, scaler, feature_cols, n_samples, label, df_real):
    """Genera n_samples sintéticos y los decodifica con el scaler."""
    samples_scaled, _ = gmm.sample(n_samples)
    samples = scaler.inverse_transform(samples_scaled)
    df_synth = pd.DataFrame(samples, columns=feature_cols)
    
    # Clamp valores discretos (variables codificadas enteras)
    int_like_cols = ['gender', 'region', 'highest_education', 'imd_band',
                     'age_band', 'disability', 'num_of_prev_attempts',
                     'studied_credits', 'module_presentation_length',
                     'n_assess', 'n_banked', 'n_interactions']
    for col in int_like_cols:
        if col in df_synth.columns:
            # Redondear y clipar al rango real
            min_v = int(df_real[col].min())
            max_v = int(df_real[col].max())
            df_synth[col] = df_synth[col].round().clip(min_v, max_v).astype(int)
    
    # Asegurar que scores estén en [0, 100]
    for col in ['score_mean', 'score_min', 'score_max']:
        if col in df_synth.columns:
            df_synth[col] = df_synth[col].clip(0, 100)
    
    # Asegurar clicks no negativos
    click_cols = [c for c in df_synth.columns if 'click' in c or 'total_' in c]
    for col in click_cols:
        df_synth[col] = df_synth[col].clip(0)
    
    df_synth['final_result'] = label
    df_synth['source'] = 'GMM_synthetic'
    return df_synth

# Generar
N_WITHDRAWN = n_synth_withdrawn if n_synth_withdrawn > 0 else len(df_withdrawn)
N_FAIL      = n_synth_fail if n_synth_fail > 0 else len(df_fail)

gmm_synth_withdrawn = generate_gmm_samples(
    gmm_withdrawn, scaler_w, FEATURE_COLS, N_WITHDRAWN, 'Withdrawn', df_withdrawn)
gmm_synth_fail = generate_gmm_samples(
    gmm_fail, scaler_f, FEATURE_COLS, N_FAIL, 'Fail', df_fail)

print(f'✅ GMM Withdrawn sintéticos: {len(gmm_synth_withdrawn)}')
print(f'✅ GMM Fail sintéticos:      {len(gmm_synth_fail)}')

In [ ]:
# Comparar distribuciones reales vs sintéticas (GMM) — sample de features clave
KEY_FEATURES = ['score_mean', 'total_clicks', 'active_days', 'num_of_prev_attempts',
                'studied_credits', 'n_assess']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(KEY_FEATURES):
    if feat not in FEATURE_COLS:
        continue
    ax = axes[i]
    idx = FEATURE_COLS.index(feat)
    
    ax.hist(df_withdrawn[feat], bins=30, alpha=0.5, color='#4ECDC4',
            label='Real', density=True, edgecolor='none')
    ax.hist(gmm_synth_withdrawn[feat], bins=30, alpha=0.5, color='#FF6B6B',
            label='GMM Synth', density=True, edgecolor='none')
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle('Distribuciones: Real vs GMM Sintéticos — Withdrawn', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gmm_distributions_withdrawn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(KEY_FEATURES):
    if feat not in FEATURE_COLS:
        continue
    ax = axes[i]
    ax.hist(df_fail[feat], bins=30, alpha=0.5, color='#4ECDC4',
            label='Real', density=True, edgecolor='none')
    ax.hist(gmm_synth_fail[feat], bins=30, alpha=0.5, color='#FFA500',
            label='GMM Synth', density=True, edgecolor='none')
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle('Distribuciones: Real vs GMM Sintéticos — Fail', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gmm_distributions_fail.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Método 2 — Variational Autoencoder (VAE)

El VAE aprende un espacio latente comprimido $z \sim \mathcal{N}(\mu, \sigma^2)$ condicionado a la clase, luego muestrea de ese espacio y decodifica para obtener nuevos ejemplos.

In [ ]:
# ── Capa de Reparametrización ──────────────────────────────────────────────
class Sampling(layers.Layer):
    """Reparametrization trick: z = mu + eps * sigma."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch  = tf.shape(z_mean)[0]
        dim    = tf.shape(z_mean)[1]
        eps    = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * eps


def build_vae(input_dim, latent_dim=16, hidden_dims=(128, 64)):
    """Construye encoder, decoder y VAE completo."""
    # ── Encoder ──
    enc_input = keras.Input(shape=(input_dim,), name='encoder_input')
    x = enc_input
    for h in hidden_dims:
        x = layers.Dense(h, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.1)(x)
    z_mean    = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
    z         = Sampling()([z_mean, z_log_var])
    encoder   = keras.Model(enc_input, [z_mean, z_log_var, z], name='encoder')

    # ── Decoder ──
    dec_input = keras.Input(shape=(latent_dim,), name='decoder_input')
    x = dec_input
    for h in reversed(hidden_dims):
        x = layers.Dense(h, activation='relu')(x)
        x = layers.BatchNormalization()(x)
    dec_output = layers.Dense(input_dim, activation='linear', name='decoder_output')(x)
    decoder    = keras.Model(dec_input, dec_output, name='decoder')

    # ── VAE completo ──
    vae_input        = keras.Input(shape=(input_dim,))
    z_mean_out, z_log_var_out, z_out = encoder(vae_input)
    reconstruction   = decoder(z_out)
    vae              = keras.Model(vae_input, reconstruction, name='vae')

    # Loss = Reconstruction + KL Divergence
    recon_loss = tf.reduce_mean(tf.square(vae_input - reconstruction), axis=1) * input_dim
    kl_loss    = -0.5 * tf.reduce_mean(1 + z_log_var_out - tf.square(z_mean_out) - tf.exp(z_log_var_out), axis=1)
    vae_loss   = tf.reduce_mean(recon_loss + kl_loss)
    vae.add_loss(vae_loss)
    vae.compile(optimizer=keras.optimizers.Adam(1e-3))

    return encoder, decoder, vae

INPUT_DIM  = len(FEATURE_COLS)
LATENT_DIM = 16
print(f'Input dim: {INPUT_DIM}, Latent dim: {LATENT_DIM}')

In [ ]:
# ── Entrenar VAE para Withdrawn ─────────────────────────────────────────────
print('=== Entrenando VAE — Withdrawn ===')

# Normalizar en [0,1] para el VAE
mm_scaler_w = MinMaxScaler()
X_w_mm = mm_scaler_w.fit_transform(df_withdrawn.values.astype(np.float32))

enc_w, dec_w, vae_w = build_vae(INPUT_DIM, LATENT_DIM)

history_w = vae_w.fit(
    X_w_mm, X_w_mm,
    epochs=200,
    batch_size=32,
    validation_split=0.1,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10, verbose=0)
    ]
)
print(f'Entrenamiento completado. Epochs: {len(history_w.history["loss"])}')

In [ ]:
# ── Entrenar VAE para Fail ──────────────────────────────────────────────────
print('=== Entrenando VAE — Fail ===')

mm_scaler_f = MinMaxScaler()
X_f_mm = mm_scaler_f.fit_transform(df_fail.values.astype(np.float32))

enc_f, dec_f, vae_f = build_vae(INPUT_DIM, LATENT_DIM)

history_f = vae_f.fit(
    X_f_mm, X_f_mm,
    epochs=200,
    batch_size=32,
    validation_split=0.1,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10, verbose=0)
    ]
)
print(f'Entrenamiento completado. Epochs: {len(history_f.history["loss"])}')

In [ ]:
# ── Curvas de entrenamiento ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, hist, title, color in zip(
    axes,
    [history_w, history_f],
    ['VAE Withdrawn', 'VAE Fail'],
    ['#FF6B6B', '#FFA500']
):
    ax.plot(hist.history['loss'], label='Train Loss', color=color, lw=2)
    ax.plot(hist.history['val_loss'], label='Val Loss', color=color, lw=2, linestyle='--')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.2)

plt.suptitle('VAE — Curvas de Entrenamiento', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vae_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Generar muestras con VAE ────────────────────────────────────────────────
def generate_vae_samples(decoder, scaler_mm, scaler_std, feature_cols,
                          n_samples, latent_dim, label, df_real):
    """Muestrea del espacio latente del VAE y decodifica."""
    z_samples = np.random.normal(0, 1, (n_samples, latent_dim)).astype(np.float32)
    decoded_mm = decoder.predict(z_samples, verbose=0)
    # De vuelta al espacio original (MinMax → StandardScaler)
    decoded = scaler_mm.inverse_transform(np.clip(decoded_mm, 0, 1))
    df_synth = pd.DataFrame(decoded, columns=feature_cols)
    
    # Postprocesado: mismos clips que en GMM
    int_like_cols = ['gender', 'region', 'highest_education', 'imd_band',
                     'age_band', 'disability', 'num_of_prev_attempts',
                     'studied_credits', 'module_presentation_length',
                     'n_assess', 'n_banked', 'n_interactions']
    for col in int_like_cols:
        if col in df_synth.columns:
            min_v = int(df_real[col].min())
            max_v = int(df_real[col].max())
            df_synth[col] = df_synth[col].round().clip(min_v, max_v).astype(int)
    
    for col in ['score_mean', 'score_min', 'score_max']:
        if col in df_synth.columns:
            df_synth[col] = df_synth[col].clip(0, 100)
    
    click_cols = [c for c in df_synth.columns if 'click' in c or 'total_' in c]
    for col in click_cols:
        df_synth[col] = df_synth[col].clip(0)
    
    df_synth['final_result'] = label
    df_synth['source'] = 'VAE_synthetic'
    return df_synth

# Generar
vae_synth_withdrawn = generate_vae_samples(
    dec_w, mm_scaler_w, scaler_w, FEATURE_COLS, N_WITHDRAWN, LATENT_DIM, 'Withdrawn', df_withdrawn)
vae_synth_fail = generate_vae_samples(
    dec_f, mm_scaler_f, scaler_f, FEATURE_COLS, N_FAIL, LATENT_DIM, 'Fail', df_fail)

print(f'✅ VAE Withdrawn sintéticos: {len(vae_synth_withdrawn)}')
print(f'✅ VAE Fail sintéticos:      {len(vae_synth_fail)}')

## 6. Evaluación de Calidad — PCA + t-SNE

In [ ]:
def evaluate_quality_pca(df_real, df_gmm, df_vae, label, feature_cols, output_dir):
    """Visualiza PCA 2D comparando real vs GMM vs VAE."""
    X_real = df_real[feature_cols].values
    X_gmm  = df_gmm[feature_cols].values
    X_vae  = df_vae[feature_cols].values
    
    n = min(500, len(X_real), len(X_gmm), len(X_vae))
    idx_r = np.random.choice(len(X_real), n, replace=False)
    idx_g = np.random.choice(len(X_gmm), n, replace=False)
    idx_v = np.random.choice(len(X_vae), n, replace=False)
    
    X_all = np.vstack([X_real[idx_r], X_gmm[idx_g], X_vae[idx_v]])
    labels_all = ['Real'] * n + ['GMM'] * n + ['VAE'] * n
    
    # PCA
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(StandardScaler().fit_transform(X_all))
    
    # t-SNE (max 1500 puntos)
    n_tsne = min(300, n)
    X_tsne_in = np.vstack([X_real[:n_tsne], X_gmm[:n_tsne], X_vae[:n_tsne]])
    labels_tsne = ['Real']*n_tsne + ['GMM']*n_tsne + ['VAE']*n_tsne
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
    X_tsne = tsne.fit_transform(StandardScaler().fit_transform(X_tsne_in))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'Real': '#4ECDC4', 'GMM': '#FF6B6B', 'VAE': '#FFE66D'}
    
    for ax, X_2d, labs, title in [
        (axes[0], X_pca, labels_all, 'PCA 2D'),
        (axes[1], X_tsne, labels_tsne, 't-SNE 2D')
    ]:
        for grp in ['Real', 'GMM', 'VAE']:
            mask = np.array(labs) == grp
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=colors[grp],
                      label=grp, alpha=0.5, s=15, edgecolors='none')
        ax.set_title(f'{title} — {label}', fontsize=12)
        ax.legend()
        ax.grid(alpha=0.15)
    
    plt.suptitle(f'Real vs Sintéticos: {label}', fontsize=14)
    plt.tight_layout()
    fname = output_dir / f'quality_{label.lower()}_pca_tsne.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Guardado: {fname.name}')

print('Generando visualizaciones de calidad...')
evaluate_quality_pca(df_withdrawn, gmm_synth_withdrawn, vae_synth_withdrawn,
                     'Withdrawn', FEATURE_COLS, OUTPUT_DIR)

In [ ]:
evaluate_quality_pca(df_fail, gmm_synth_fail, vae_synth_fail,
                     'Fail', FEATURE_COLS, OUTPUT_DIR)

## 7. Estadísticas Descriptivas — Comparativa Real vs Sintéticos

In [ ]:
def compare_stats(df_real, df_gmm, df_vae, feature_cols, label):
    """Tabla comparativa de medias y stds."""
    num_cols = [c for c in feature_cols
                if c in df_real.select_dtypes(include=[np.number]).columns]
    
    stats = pd.DataFrame()
    stats['Real_mean']  = df_real[num_cols].mean()
    stats['Real_std']   = df_real[num_cols].std()
    stats['GMM_mean']   = df_gmm[num_cols].mean()
    stats['GMM_std']    = df_gmm[num_cols].std()
    stats['VAE_mean']   = df_vae[num_cols].mean()
    stats['VAE_std']    = df_vae[num_cols].std()
    
    # Delta relativo
    stats['GMM_delta%'] = ((stats['GMM_mean'] - stats['Real_mean']) / (stats['Real_mean'].abs() + 1e-8) * 100).round(1)
    stats['VAE_delta%'] = ((stats['VAE_mean'] - stats['Real_mean']) / (stats['Real_mean'].abs() + 1e-8) * 100).round(1)
    
    print(f'\n=== Comparativa Estadística — {label} ===')
    print(stats.round(3).to_string())
    return stats

stats_w = compare_stats(df_withdrawn, gmm_synth_withdrawn, vae_synth_withdrawn,
                        FEATURE_COLS, 'Withdrawn')

In [ ]:
stats_f = compare_stats(df_fail, gmm_synth_fail, vae_synth_fail,
                        FEATURE_COLS, 'Fail')

## 8. Guardar Resultados

In [ ]:
# Añadir source a datos reales
df_withdrawn_save = df_withdrawn.copy()
df_withdrawn_save['final_result'] = 'Withdrawn'
df_withdrawn_save['source'] = 'real'

df_fail_save = df_fail.copy()
df_fail_save['final_result'] = 'Fail'
df_fail_save['source'] = 'real'

# Guardar sintéticos individuales
gmm_synth_withdrawn.to_csv(OUTPUT_DIR / 'synthetic_gmm_withdrawn.csv', index=False)
gmm_synth_fail.to_csv(OUTPUT_DIR / 'synthetic_gmm_fail.csv', index=False)
vae_synth_withdrawn.to_csv(OUTPUT_DIR / 'synthetic_vae_withdrawn.csv', index=False)
vae_synth_fail.to_csv(OUTPUT_DIR / 'synthetic_vae_fail.csv', index=False)

# Guardar estadísticas
stats_w.to_csv(OUTPUT_DIR / 'stats_comparison_withdrawn.csv')
stats_f.to_csv(OUTPUT_DIR / 'stats_comparison_fail.csv')

print('✅ Archivos individuales guardados')

In [ ]:
# ── Dataset combinado: real + sintéticos GMM + VAE ─────────────────────────
combined = pd.concat([
    df_withdrawn_save,
    df_fail_save,
    gmm_synth_withdrawn,
    gmm_synth_fail,
    vae_synth_withdrawn,
    vae_synth_fail,
], ignore_index=True)

combined.to_csv(OUTPUT_DIR / 'synthetic_combined.csv', index=False)

print('=== Dataset Combinado ===')
print(combined.groupby(['final_result', 'source']).size().to_string())

print(f'\n✅ synthetic_combined.csv guardado: {len(combined)} filas')

In [ ]:
# ── Resumen final ────────────────────────────────────────────────────────────
print('='*60)
print('RESUMEN GENERACIÓN DE DATOS SINTÉTICOS')
print('='*60)
print(f'  Reales Withdrawn:        {len(df_withdrawn):>6}')
print(f'  Reales Fail:             {len(df_fail):>6}')
print(f'  GMM Sintéticos Withdrawn:{len(gmm_synth_withdrawn):>6}')
print(f'  GMM Sintéticos Fail:     {len(gmm_synth_fail):>6}')
print(f'  VAE Sintéticos Withdrawn:{len(vae_synth_withdrawn):>6}')
print(f'  VAE Sintéticos Fail:     {len(vae_synth_fail):>6}')
print(f'  TOTAL combinado:         {len(combined):>6}')
print()
print(f'📁 Archivos en: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')

## 9. Visualización Final — Distribución Total del Dataset Aumentado

In [ ]:
# Clase original completa + sintéticos
pass_df  = full_df[full_df['final_result'] == 'Pass'][FEATURE_COLS].copy()
pass_df['final_result'] = 'Pass'
pass_df['source'] = 'real'

dist_df = full_df[full_df['final_result'] == 'Distinction'][FEATURE_COLS].copy()
dist_df['final_result'] = 'Distinction'
dist_df['source'] = 'real'

augmented = pd.concat([pass_df, dist_df, combined], ignore_index=True)
counts = augmented.groupby(['final_result', 'source']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de distribución
ax = axes[0]
result_counts = augmented['final_result'].value_counts()
colors_bar = ['#4ECDC4', '#95E1D3', '#FF6B6B', '#FFA500']
bars = ax.bar(result_counts.index, result_counts.values,
               color=colors_bar[:len(result_counts)], edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, result_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontsize=10, color='white')
ax.set_title('Distribución Dataset Aumentado', fontsize=13)
ax.set_ylabel('Nº Estudiantes')
ax.grid(axis='y', alpha=0.2)

# Proporción real vs sintético
ax2 = axes[1]
source_lbl = ['Real (todos)', 'GMM Synth', 'VAE Synth']
source_cnt = [
    (augmented['source'] == 'real').sum(),
    (augmented['source'] == 'GMM_synthetic').sum(),
    (augmented['source'] == 'VAE_synthetic').sum(),
]
wedge_colors = ['#4ECDC4', '#FF6B6B', '#FFE66D']
wedges, texts, autotexts = ax2.pie(source_cnt, labels=source_lbl,
                                    autopct='%1.1f%%', colors=wedge_colors,
                                    startangle=90, pctdistance=0.8)
for t in texts + autotexts:
    t.set_color('white')
ax2.set_title('Proporción Real vs Sintético', fontsize=13)

plt.suptitle('Dataset Final Aumentado con Datos Sintéticos', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'final_augmented_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Notebook completado. Todos los archivos en:\n{OUTPUT_DIR}')